In [ ]:
import numpy as np
from tensorflow.keras.datasets import mnist

np.random.seed(42)

# Load MNIST

In [ ]:
import numpy as np
from tensorflow.keras.datasets import mnist

(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Normalize
X_train = X_train.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0

# Add channel dimension
X_train = X_train.reshape(-1, 1, 28, 28)
X_test = X_test.reshape(-1, 1, 28, 28)

# Reduce dataset
X_train_small = X_train[:1000]
y_train_small = y_train[:1000]

X_test_small = X_test[:200]
y_test_small = y_test[:200]

print("Train shape:", X_train_small.shape)

Train shape: (1000, 1, 28, 28)


# Shape planning

- **Input**: (1,28,28)
- **After Convo(3x3)**: (8,26,26)
- **After Pool**: (8,13,13)
- **After Conv**: (16,11,11)
- **After Pool**: (16,5,5)
- **Flatten**:  16 x 5 x 5 = 400
- **So FC becomes**: Linear(400,10)

# Linear layer

In [ ]:
class Linear:
    def __init__(self, in_features, out_features):
        self.W = np.random.randn(in_features, out_features) * np.sqrt(2. / in_features)
        self.b = np.zeros((1, out_features))
        self.X = None

    def forward(self, X):
        self.X = X
        return X @ self.W + self.b

    def backward(self, d_out):
        self.dW = self.X.T @ d_out
        self.db = np.sum(d_out, axis=0, keepdims=True)
        return d_out @ self.W.T

# ReLU(Rectified Linear Unit)

### ReLU is :  f(x) = max(0,x)

- It introduces non-linearity
- Without it, the entire network is just big linear function


In [ ]:
class ReLU:
    def __init__(self):
        self.X = None

    def forward(self, X):
        self.X = X
        return np.maximum(0, X)

    def backward(self, d_out):
        dX = d_out.copy()
        dX[self.X <= 0] = 0
        return dX

# Softmax + CrossEntropy

### This layer:
- Converts raw scores to probabilities
- Computes classification loss
- Produces first gradient for backpropagation


In [ ]:
class SoftmaxCrossEntropy:
    def __init__(self):
        self.probs = None
        self.y_true = None

    def forward(self, logits, y_true):
        shifted = logits - np.max(logits, axis=1, keepdims=True)
        exp_scores = np.exp(shifted)
        self.probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

        self.y_true = y_true
        batch_size = logits.shape[0]

        loss = -np.log(self.probs[range(batch_size), y_true])
        return np.mean(loss)

    def backward(self):
        batch_size = self.probs.shape[0]
        d_logits = self.probs.copy()
        d_logits[range(batch_size), self.y_true] -= 1
        return d_logits / batch_size

This layer is used at the **end of the network** for classification.

### What this layer outputs:

#### Forward:
- Returns Scalar loss

#### Backward:
- Returns gradient matrix:

```
(batch_size, num_classes)
```
#### Which flows into last Linear layer


---


## 1. Softmax

The model gives raw scores like:

[2.3, 1.1, 0.2]

Softmax converts them into probabilities like:

[0.70, 0.20, 0.10]

Properties:
- Each value is between 0 and 1
- All values in a row sum to 1
- The highest value represents the predicted class

---

## 2. Cross Entropy Loss

We only care about the probability of the correct class.

Loss formula (in simple terms):

Loss = -log(probability of correct class)

Meaning:
- If correct class probability is high - loss is small
- If correct class probability is low - loss is large

The final loss is the average across all samples in the batch.

---

## Backward Pass (Gradient)

The gradient simplifies to:

gradient = probabilities - true_labels

What this means:
- If prediction is correct - small gradient
- If prediction is wrong - larger gradient
- This gradient is sent back to update the weights

---

## Why We Combine Softmax + Cross Entropy

- Cleaner math
- More stable computation
- This is how real deep learning frameworks implement it

## Conv2D Layer

The convolution layer extracts spatial features from images.

It:
- Slides small filters (kernels) across the input
- Performs element-wise multiplication
- Sums the result
- Produces feature maps

Backward pass computes gradients for:
- Filter weights
- Bias
- Input


In [ ]:
class Conv2D:
    def __init__(self, in_channels, out_channels, kernel_size):
        self.kernel_size = kernel_size
        self.out_channels = out_channels

        self.W = np.random.randn(out_channels, in_channels, kernel_size, kernel_size) \
                 * np.sqrt(2.0 / (in_channels * kernel_size * kernel_size))
        self.b = np.zeros((out_channels, 1))

        self.X = None

    def forward(self, X):
        self.X = X
        batch_size, in_channels, height, width = X.shape
        k = self.kernel_size

        out_h = height - k + 1
        out_w = width - k + 1

        out = np.zeros((batch_size, self.out_channels, out_h, out_w))

        for n in range(batch_size):
            for f in range(self.out_channels):
                for i in range(out_h):
                    for j in range(out_w):
                        region = X[n, :, i:i+k, j:j+k]
                        out[n, f, i, j] = np.sum(region * self.W[f]) + self.b[f, 0]

        return out

    def backward(self, d_out):
        batch_size, _, out_h, out_w = d_out.shape
        _, in_channels, height, width = self.X.shape
        k = self.kernel_size

        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)
        dX = np.zeros_like(self.X)

        for n in range(batch_size):
            for f in range(self.out_channels):
                for i in range(out_h):
                    for j in range(out_w):

                        region = self.X[n, :, i:i+k, j:j+k]

                        self.dW[f] += region * d_out[n, f, i, j]
                        self.db[f, 0] += d_out[n, f, i, j]
                        dX[n, :, i:i+k, j:j+k] += self.W[f] * d_out[n, f, i, j]

        return dX

## Qucik Shape test

In [ ]:
conv_test = Conv2D(1, 8, 3)
X_dummy = np.random.randn(2, 1, 28, 28)
out = conv_test.forward(X_dummy)

print("Conv output shape:", out.shape)

Conv output shape: (2, 8, 26, 26)


## MaxPooling Layer

MaxPooling reduces spatial size.

It:
- Divides input into 2×2 regions
- Takes maximum value from each region
- Reduces width and height by half

Backward pass:
- Sends gradient only to positions that had the maximum value

In [ ]:
class MaxPool2D:
    def __init__(self, pool_size=2):
        self.pool_size = pool_size
        self.X = None
        self.mask = None

    def forward(self, X):
        self.X = X
        batch_size, channels, height, width = X.shape
        p = self.pool_size

        out_h = height // p
        out_w = width // p

        out = np.zeros((batch_size, channels, out_h, out_w))
        self.mask = np.zeros_like(X)

        for n in range(batch_size):
            for c in range(channels):
                for i in range(out_h):
                    for j in range(out_w):

                        region = X[n, c, i*p:(i+1)*p, j*p:(j+1)*p]
                        max_val = np.max(region)
                        out[n, c, i, j] = max_val

                        mask = (region == max_val)
                        self.mask[n, c, i*p:(i+1)*p, j*p:(j+1)*p] = mask

        return out

    def backward(self, d_out):
        batch_size, channels, out_h, out_w = d_out.shape
        p = self.pool_size

        dX = np.zeros_like(self.X)

        for n in range(batch_size):
            for c in range(channels):
                for i in range(out_h):
                    for j in range(out_w):

                        dX[n, c, i*p:(i+1)*p, j*p:(j+1)*p] += \
                            self.mask[n, c, i*p:(i+1)*p, j*p:(j+1)*p] * d_out[n, c, i, j]

        return dX

## Quick Test

In [ ]:
pool_test = MaxPool2D(2)
out = pool_test.forward(out)

print("Pool output shape:", out.shape)

Pool output shape: (2, 8, 13, 13)


## Full CNN on MNIST (SGD)

We are assembling the full Convolutional Neural Network using:

Conv2D -> ReLU -> MaxPool  
Conv2D -> ReLU -> MaxPool  
Flatten  
Fully Connected  
Softmax + Cross Entropy  

This network:

- Extracts spatial features using convolution
- Reduces spatial dimensions using pooling
- Classifies digits using a fully connected layer
- Trains using manual backpropagation

In [ ]:
#Initialize CNN
conv1 = Conv2D(1, 8, 3)
relu1 = ReLU()
pool1 = MaxPool2D(2)

conv2 = Conv2D(8, 16, 3)
relu2 = ReLU()
pool2 = MaxPool2D(2)

fc = Linear(16 * 5 * 5, 10)
loss_fn = SoftmaxCrossEntropy()

learning_rate = 0.01

# Training loop

In [ ]:
epochs = 5
batch_size = 50

for epoch in range(epochs):

    epoch_loss = 0

    for i in range(0, 1000, batch_size):

        X_batch = X_train_small[i:i+batch_size]
        y_batch = y_train_small[i:i+batch_size]

        # -------- Forward --------
        out = conv1.forward(X_batch)
        out = relu1.forward(out)
        out = pool1.forward(out)

        out = conv2.forward(out)
        out = relu2.forward(out)
        out = pool2.forward(out)

        out_flat = out.reshape(out.shape[0], -1)
        logits = fc.forward(out_flat)

        loss = loss_fn.forward(logits, y_batch)
        epoch_loss += loss

        # -------- Backward --------
        d_logits = loss_fn.backward()
        d_out = fc.backward(d_logits)
        d_out = d_out.reshape(out.shape[0], 16, 5, 5)

        d_out = pool2.backward(d_out)
        d_out = relu2.backward(d_out)
        d_out = conv2.backward(d_out)

        d_out = pool1.backward(d_out)
        d_out = relu1.backward(d_out)
        d_out = conv1.backward(d_out)

        # -------- Update --------
        for layer in [conv1, conv2]:
            layer.W -= learning_rate * layer.dW
            layer.b -= learning_rate * layer.db

        fc.W -= learning_rate * fc.dW
        fc.b -= learning_rate * fc.db

    print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}")

Epoch 1, Loss: 43.3711
Epoch 2, Loss: 36.3677
Epoch 3, Loss: 29.9189
Epoch 4, Loss: 24.2305
Epoch 5, Loss: 19.9617


## Evaluate

In [ ]:
out = conv1.forward(X_test_small)
out = relu1.forward(out)
out = pool1.forward(out)

out = conv2.forward(out)
out = relu2.forward(out)
out = pool2.forward(out)

out_flat = out.reshape(out.shape[0], -1)
logits = fc.forward(out_flat)

predictions = np.argmax(logits, axis=1)
accuracy = np.mean(predictions == y_test_small)

print("Test Accuracy:", accuracy)

Test Accuracy: 0.675


## Implementing Adam Optimizer

Adam (Adaptive Moment Estimation) improves training by:

1. Keeping a running average of gradients (momentum).
2. Keeping a running average of squared gradients.
3. Applying bias correction.
4. Updating parameters adaptively.

Why use Adam?

- Faster convergence
- More stable training
- Less sensitive to learning rate
- Widely used in deep learning

We will replace simple SGD with Adam.

Adam Formula (Simple Understanding)

For each parameter:

- Compute gradient
- Update first moment (m)
- Update second moment (v)
- Bias correction
- Update weights

# Adam Class

In [ ]:
class Adam:
    def __init__(self, parameters, lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.parameters = parameters
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon

        self.m = [np.zeros_like(p) for p in parameters]
        self.v = [np.zeros_like(p) for p in parameters]
        self.t = 0

    def step(self, grads):
        self.t += 1

        for i, (p, g) in enumerate(zip(self.parameters, grads)):

            # Update biased first moment estimate
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g

            # Update biased second moment estimate
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * (g ** 2)

            # Bias correction
            m_hat = self.m[i] / (1 - self.beta1 ** self.t)
            v_hat = self.v[i] / (1 - self.beta2 ** self.t)

            # Update parameters
            p -= self.lr * m_hat / (np.sqrt(v_hat) + self.epsilon)

### What This Is Doing

Instead of:
```
W = W - lr * gradient
```

Adam does:
```
Adaptive learning rate per parameter
Momentum smoothing
Bias correction
```
Much more stable.

## Initialize Adam for CNN

Replacing old learning rate update logic

In [ ]:
parameters = [
    conv1.W, conv1.b,
    conv2.W, conv2.b,
    fc.W, fc.b
]

optimizer = Adam(parameters, lr=0.001)

In [ ]:
conv1 = Conv2D(1, 8, 3)
relu1 = ReLU()
pool1 = MaxPool2D(2)

conv2 = Conv2D(8, 16, 3)
relu2 = ReLU()
pool2 = MaxPool2D(2)

fc = Linear(16 * 5 * 5, 10)
loss_fn = SoftmaxCrossEntropy()

parameters = [
    conv1.W, conv1.b,
    conv2.W, conv2.b,
    fc.W, fc.b
]

optimizer = Adam(parameters, lr=0.001)

# Now changing the weight update section in training loop with

```
grads = [
    conv1.dW, conv1.db,
    conv2.dW, conv2.db,
    fc.dW, fc.db
]

optimizer.step(grads)
```

In [ ]:
epochs = 5
batch_size = 50

for epoch in range(epochs):

    epoch_loss = 0

    for i in range(0, 1000, batch_size):

        X_batch = X_train_small[i:i+batch_size]
        y_batch = y_train_small[i:i+batch_size]

        # -------- Forward --------
        out = conv1.forward(X_batch)
        out = relu1.forward(out)
        out = pool1.forward(out)

        out = conv2.forward(out)
        out = relu2.forward(out)
        out = pool2.forward(out)

        out_flat = out.reshape(out.shape[0], -1)
        logits = fc.forward(out_flat)

        loss = loss_fn.forward(logits, y_batch)
        epoch_loss += loss

        # -------- Backward --------
        d_logits = loss_fn.backward()
        d_out = fc.backward(d_logits)
        d_out = d_out.reshape(out.shape[0], 16, 5, 5)

        d_out = pool2.backward(d_out)
        d_out = relu2.backward(d_out)
        d_out = conv2.backward(d_out)

        d_out = pool1.backward(d_out)
        d_out = relu1.backward(d_out)
        d_out = conv1.backward(d_out)

        grads = [
          conv1.dW, conv1.db,
          conv2.dW, conv2.db,
          fc.dW, fc.db
        ]

        optimizer.step(grads)

    print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}")



Epoch 1, Loss: 42.9505
Epoch 2, Loss: 32.0299
Epoch 3, Loss: 21.9052
Epoch 4, Loss: 14.6230
Epoch 5, Loss: 10.5838


# Re-Evaluate again

In [ ]:
out = conv1.forward(X_test_small)
out = relu1.forward(out)
out = pool1.forward(out)

out = conv2.forward(out)
out = relu2.forward(out)
out = pool2.forward(out)

out_flat = out.reshape(out.shape[0], -1)
logits = fc.forward(out_flat)

predictions = np.argmax(logits, axis=1)
accuracy = np.mean(predictions == y_test_small)

print("Test Accuracy:", accuracy)

Test Accuracy: 0.81


## Optimizer Comparison (SGD vs Adam)

We trained the same CNN architecture using SGD and Adam.

Results after 5 epochs:

SGD:
- Final Loss = 19.96
- Test Accuracy = 67.5%

Adam:
- Final Loss = 10.58
- Test Accuracy = 81%

Adam converged faster and achieved significantly higher accuracy.
This demonstrates the benefit of adaptive optimization methods
for deep neural networks.

## Final Conclusion – CNN from Scratch on MNIST

In this experiment, we successfully implemented a Convolutional Neural Network (CNN) entirely from scratch using NumPy.

The following components were built manually:

- Conv2D layer (forward and backward propagation)
- MaxPooling layer (forward and backward propagation)
- ReLU activation
- Fully Connected (Linear) layer
- Softmax with Cross-Entropy loss
- Stochastic Gradient Descent (SGD) optimizer
- Adam optimizer

We trained the network on 1000 MNIST samples and evaluated it on a test subset.

### Results:

SGD:
- Final Loss = 19.96
- Test Accuracy = 67.5%

Adam:
- Final Loss = 10.58
- Test Accuracy = 81%

### Observations:

- Adam converged faster than SGD.
- Adam achieved significantly higher accuracy in the same number of epochs.
- The decreasing loss confirms that backpropagation was correctly implemented.
- The model successfully learned meaningful spatial features from images.

### Key Takeaways:

- Manual implementation helps deeply understand gradient flow in CNNs.
- Optimizer choice has a strong impact on convergence speed and performance.
- Even a simple 2-layer CNN can perform well on MNIST when implemented correctly.

This experiment validates the correctness of the CNN architecture and backpropagation implementation.